# 개별종목 조합K — RandomForest

`기본모델/02.RandomForest.ipynb`과 같은 `models.random_forest.build_random_forest_baseline`을 가져오고
조합K 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.random_forest import build_random_forest_baseline  # noqa: E402

MODEL_NAME = 'RandomForest'
MODEL_BUILDER = build_random_forest_baseline


In [2]:
# 2. 조합K의 피처 값만 지정합니다.
import json

COMBINATION = 'K'
FEATURE_COLUMNS = (
    'atr_ratio',
    'bb_bandwidth',
    'hv_regime',
    'five_day_return',
    'relative_ret_5_market',
    'sma_gap_5_20',
    'sma_gap_20_60',
    'rsi_14',
    'macd_hist_ratio',
    'bb_position',
    'hv_20',
    'vol_ratio_20',
    'obv_slope_20',
    'daily_return',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합K 피처: ('atr_ratio', 'bb_bandwidth', 'hv_regime', 'five_day_return', 'relative_ret_5_market', 'sma_gap_5_20', 'sma_gap_20_60', 'rsi_14', 'macd_hist_ratio', 'bb_position', 'hv_20', 'vol_ratio_20', 'obv_slope_20', 'daily_return')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4487,0.5012,-0.0525,0.3536,0.3627,0.0543,0.3679,0.2152,0.3092
1,2,balanced,980,20150123,20150421,0.3864,0.3978,-0.0115,0.3625,0.3688,0.0586,0.3703,0.2875,0.3399
2,3,NaN,1210,20151228,20160328,0.3672,0.3762,-0.0089,0.3614,0.3622,0.0459,0.3604,0.3565,0.3617
3,4,balanced,1439,20161202,20170228,0.4259,0.4617,-0.0358,0.3700,0.3752,0.0720,0.3864,0.2675,0.3413
4,5,balanced,1669,20171113,20180207,0.3901,0.3901,0.0000,0.3662,0.3723,0.0640,0.3691,0.3349,0.3623
5,6,balanced,1899,20181024,20190118,0.4096,0.3725,0.0371,0.4093,0.4113,0.1186,0.4129,0.4272,0.4152
6,7,balanced,2129,20190930,20191224,0.4252,0.4781,-0.0529,0.3516,0.3623,0.0593,0.3747,0.2630,0.3334
7,8,balanced,2359,20200902,20201130,0.3667,0.3476,0.0190,0.3666,0.3733,0.0603,0.3782,0.4530,0.3915
8,9,balanced,2589,20210806,20211105,0.3770,0.3914,-0.0144,0.3679,0.3744,0.0598,0.3757,0.3219,0.3539
9,10,balanced,2818,20220714,20221012,0.3654,0.3454,0.0200,0.3627,0.3649,0.0471,0.3742,0.3263,0.3505


,OOS 폴드 평균
accuracy,0.3950
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0019
macro_f1,0.3701
balanced_accuracy,0.3748
mcc,0.0667
pr_auc_macro_ovr,0.3784
down_recall,0.3401
core_harmonic_mean,0.3624


재실행 명령: python scripts/run_stock_model_experiment.py
